# OCI Generative AI + xAI Tools basic usage

This notebook uses the OCI Generative AI OpenAI-compatible endpoint. Configure `OCI_GENAI_BASE_URL`, `OCI_GENAI_KEY`, and `AGENT_MODEL_ID` in the repository root `.env` file before running it.

Reference: https://docs.oracle.com/en-us/iaas/Content/generative-ai/get-started-agents.htm

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

base_url = os.environ["OCI_GENAI_BASE_URL"]
model = os.getenv("AGENT_MODEL_ID", "xai.grok-4.3")

client = OpenAI(
    api_key=os.environ["OCI_GENAI_KEY"],
    base_url=base_url,
)

def save_response(response, filename):
    with open(filename, "w", encoding="utf-8") as f:
        is_delta = False
        for event in response:
            if event.type.endswith(".delta"):
                if not is_delta:
                    f.write("\n" + str(event) + "\n")
                else:
                    f.write(str(event.delta))
                is_delta = True
            else:
                f.write("\n" + str(event) + "\n")
                is_delta = False

                if event.type == "response.output_item.done":
                    f.write("\n\n" + "*" * 50 + "\n\n")

# x_search

`x_search` lets the model search recent public X posts. Useful parameters include `allowed_x_handles`, `excluded_x_handles`, `from_date`, `to_date`, `enable_image_understanding`, and `enable_video_understanding`.

Reference: https://docs.x.ai/developers/tools/x-search

In [ ]:
response = client.responses.create(
    model=model,
    tools=[{"type": "x_search"}],
    input="Give me a very short summary of the last 3 posts by @elonmusk.",
    stream=True,
)

save_response(response, "x_search_response.txt")

# code_interpreter

`code_interpreter` lets the model write and run Python code for calculations, data analysis, and tasks that need execution rather than text-only reasoning.

Reference: https://docs.x.ai/developers/tools/code-execution

In [ ]:
response = client.responses.create(
    model=model,
    tools=[{"type": "code_interpreter"}],
    reasoning={"effort": "low"},
    input=[
        {
            "role": "user",
            "content": "Calculate how many prime numbers are between 12345 and 67890.",
        },
    ],
    stream=True,
)

save_response(response, "code_interpreter_response.txt")